# Merit Order Model 

This model is built to simulate how the UK National Grid clears the day-ahead wholesale electricity market based on demand and marginal costs. 

Match hourly demand with the cheapest available power supply. 

Set i as power plant type
each has capacity and a short-run marginal cost (SRMC)

SRMC = Fuel price + carbon price, plant effiency, carbon intensity $EF_i$ and Variable from operation and maintenance (VOM) 

$$SRMC_i = \frac{P_{fuel}}{Plant \;efficiency}+P_{carbon}\times EF_i + VOM_i$$

Sort the supply or generator from cheapest to highest. 

- Renewables have a variable cost of 0 - > so they sit at the bottom of the stack model
- Nuclear in the middle stack 
- Oik and GAS (CCGT)

Add later
Import power from France, Belgium, Norway and Netherland 
Availability factor of Wind and Solar power 
_ how to account for the Boundary constraints - > should I model North and south separately. 


In [1]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 

# Define UK Grid demand for the target hour 
uk_demand = 40000  # in MW

# Set capacity and marginal costs for each power plants
generation_fleet = { 
                    'Solar': {'capacity': 20000, 'marginal_cost': 50},
                    'Wind': {'capacity': 15000, 'marginal_cost': 70},
                    'Nuclear': {'capacity': 10000, 'marginal_cost': 30},
                    'Biomass': {'capacity': 5000, 'marginal_cost': 0},
                    'Gas (CCGT)': {'capacity': 5000, 'marginal_cost': 0},
                    'Oil': {'capacity': 5000, 'marginal_cost': 0},
                   }

# Build the merit order dataframe 
df = pd.DataFrame.from_dict(generation_fleet, orient='index')
df.index.name = 'Technology'
df = df.reset_index()
df = df.sort_values(by='marginal_cost')
df['cumulative_capacity'] = df['capacity'].cumsum()
df['market_clearing_price'] = df['marginal_cost'].where(df['cumulative_capacity'] >= uk_grid_demand).ffill()
df = df[df['market_clearing_price'].notna()]


dispatched_power = 0
market_clearing_price = 0 
clearing_asset = "None"

for asset, row in df.iterrows():
    if row["cumulative_capacity"] < uk_demand:
        # this asset is fully dispatched
        dispatched_power += row['cumulative_capacity']
    else:
        market_clearing_price = row['marginal_cost']
        clearing_asset = row['Technology']
        
        
        
print(f"Cleared Asset: {clearing_asset}")
print(f"Simulated UK Electricity Market Clearing Price: £{market_clearing_price}/MWh")

NameError: name 'uk_grid_demand' is not defined